In [1]:
# Cell 1 — Config
SEED = 42

# Paths (change to your actual CSV paths)
TASHBEEH_CSV = "tashbeeh_dataset.csv"
NON_TASHBEEH_CSV = "non_tashbeeh_random_sample.csv"

TEXT_COL = "البيت_الكامل"   # or "التعبير البلاغي" if you prefer
LABEL_COL = "label"

# Split
TEST_SIZE = 0.15
VAL_SIZE  = 0.15

# Model choice (hardcode one of these)
MODEL_ID = 4
MODEL_CANDIDATES = {
    1: "aubmindlab/bert-base-arabertv02",
    2: "CAMeL-Lab/bert-base-arabic-camelbert-mix",
    3: "CAMeL-Lab/bert-base-arabic-camelbert-ca",
    4: "aubmindlab/bert-large-arabertv2"
}
MODEL_NAME = MODEL_CANDIDATES[MODEL_ID]

# Training
MAX_LEN = 96
BATCH_SIZE = 8
EPOCHS = 20
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06

OUTPUT_DIR = f"runs_tashbeeh/{MODEL_NAME.replace('/','_')}"
print("Model:", MODEL_NAME)
print("Output:", OUTPUT_DIR)


Model: aubmindlab/bert-large-arabertv2
Output: runs_tashbeeh/aubmindlab_bert-large-arabertv2


In [2]:
# Cell 2 — Imports
import os, random
import numpy as np
import pandas as pd

import torch
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

import evaluate


C:\Users\USER\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\USER\miniconda3\envs\torch-gpu\lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
c:\Users\USER\miniconda3\envs\torch-gpu\lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
c:\Users\USER\miniconda3\envs\torch-gpu\lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytr

In [3]:
# Cell 3 — Repro + Device
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| cuda:", torch.cuda.is_available())


device: cuda | cuda: True


In [4]:
# Cell 4 — Load + Label
df_pos = pd.read_csv(TASHBEEH_CSV)
df_neg = pd.read_csv(NON_TASHBEEH_CSV)

df_pos = df_pos[[TEXT_COL]].copy()
df_neg = df_neg[[TEXT_COL]].copy()

df_pos[LABEL_COL] = 1
df_neg[LABEL_COL] = 0

df = pd.concat([df_pos, df_neg], ignore_index=True)
df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
df = df[df[TEXT_COL].str.len() > 0].reset_index(drop=True)

print("Total:", df.shape)
print(df[LABEL_COL].value_counts())
df.head()


Total: (1182, 2)
label
1    591
0    591
Name: count, dtype: int64


,البيت_الكامل,label
0,أهُمُّ بِشَيْءٍ واللَّيالي كأنَّهَا تُطارِدُنِ...,1
1,تَثَنَّى عَلى قَدْرِ الطِّعانِ كأنَّما مَفاصِل...,1
2,وَلمَّا رأيْتُ النَّاسَ دُونَ مَحَلِّهِ تَيَقّ...,1
3,مُخَضَّبَةٌ والْقَوْمُ صَرْعَى كأنَّها وَإنْ ل...,1
4,تُنَكِّسِهُمْ والسَّابِقاتُ جِبالُهُمْ وَتَطْع...,1


In [5]:
# Cell — Train/Val/Test split (stratified) with ClassLabel casting
from datasets import Features, Value, ClassLabel

# Build Dataset with explicit features (label as ClassLabel)
features = Features({
    TEXT_COL: Value("string"),
    LABEL_COL: ClassLabel(names=["non_tashbeeh", "tashbeeh"]),
})

full_ds = Dataset.from_pandas(
    df[[TEXT_COL, LABEL_COL]].reset_index(drop=True),
    features=features
)

# Split out test
tmp = full_ds.train_test_split(
    test_size=TEST_SIZE,
    seed=SEED,
    stratify_by_column=LABEL_COL
)
train_val_ds, test_ds = tmp["train"], tmp["test"]

# Split train/val
val_relative = VAL_SIZE / (1.0 - TEST_SIZE)
tmp2 = train_val_ds.train_test_split(
    test_size=val_relative,
    seed=SEED,
    stratify_by_column=LABEL_COL
)
train_ds, val_ds = tmp2["train"], tmp2["test"]

print("Train:", len(train_ds), "Val:", len(val_ds), "Test:", len(test_ds))

def balance(ds):
    y = np.array(ds[LABEL_COL])
    return {0: int((y==0).sum()), 1: int((y==1).sum())}

print("Train balance:", balance(train_ds))
print("Val balance  :", balance(val_ds))
print("Test balance :", balance(test_ds))
print("Label names:", full_ds.features[LABEL_COL].names)


Train: 826 Val: 178 Test: 178
Train balance: {0: 413, 1: 413}
Val balance  : {0: 89, 1: 89}
Test balance : {0: 89, 1: 89}
Label names: ['non_tashbeeh', 'tashbeeh']


In [6]:
# Cell 5 — Tokenize train/val/test (using train_ds/val_ds/test_ds)
# ---------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(
        batch[TEXT_COL],
        truncation=True,
        max_length=MAX_LEN,
    )

# Tokenize; keep label column for now
train_tok = train_ds.map(tokenize_batch, batched=True)
val_tok   = val_ds.map(tokenize_batch, batched=True)
test_tok  = test_ds.map(tokenize_batch, batched=True)

# Remove text column after tokenization to save memory
train_tok = train_tok.remove_columns([TEXT_COL])
val_tok   = val_tok.remove_columns([TEXT_COL])
test_tok  = test_tok.remove_columns([TEXT_COL])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Trainer expects label column to be named "labels"
train_tok = train_tok.rename_column(LABEL_COL, "labels")
val_tok   = val_tok.rename_column(LABEL_COL, "labels")
test_tok  = test_tok.rename_column(LABEL_COL, "labels")

# Make sure dataset returns torch tensors
train_tok.set_format(type="torch")
val_tok.set_format(type="torch")
test_tok.set_format(type="torch")

print("Keys:", train_tok.column_names)
print("Example:", {k: (v.shape if hasattr(v, "shape") else v) for k, v in train_tok[0].items()})


c:\Users\USER\miniconda3\envs\torch-gpu\lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Map: 100%|██████████| 178/178 [00:00<00:00, 15429.80 examples/s]

Keys: ['labels', 'input_ids', 'token_type_ids', 'attention_mask']
Example: {'labels': torch.Size([]), 'input_ids': torch.Size([13]), 'token_type_ids': torch.Size([13]), 'attention_mask': torch.Size([13])}


In [7]:
# Cell 6 — Metrics (no sklearn)
import numpy as np
import evaluate

acc = evaluate.load("accuracy")
prec = evaluate.load("precision")
rec = evaluate.load("recall")
f1 = evaluate.load("f1")
cm = evaluate.load("confusion_matrix")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    out = {}
    out["accuracy"] = acc.compute(predictions=preds, references=labels)["accuracy"]
    out["precision_pos"] = prec.compute(predictions=preds, references=labels, average="binary")["precision"]
    out["recall_pos"] = rec.compute(predictions=preds, references=labels, average="binary")["recall"]
    out["f1_pos"] = f1.compute(predictions=preds, references=labels, average="binary")["f1"]

    cmat = cm.compute(predictions=preds, references=labels, labels=[0, 1])["confusion_matrix"]
    out["tn"], out["fp"] = int(cmat[0][0]), int(cmat[0][1])
    out["fn"], out["tp"] = int(cmat[1][0]), int(cmat[1][1])
    return out


In [8]:
# Cell 6.5 — Build model (Trainer will move it to GPU automatically)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)


c:\Users\USER\miniconda3\envs\torch-gpu\lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at aubmindlab/bert-large-arabertv2 and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
# Cell 7 — TrainingArguments + Trainer (transformers 4.35.2)
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,

    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_pos",
    greater_is_better=True,

    logging_strategy="steps",
    logging_steps=50,
    report_to="none",

    fp16=torch.cuda.is_available(),   # True on GPU
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
print("Trainer ready.")


c:\Users\USER\miniconda3\envs\torch-gpu\lib\site-packages\accelerate\accelerator.py:439: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Trainer ready.


In [10]:
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("torch cuda version:", torch.version.cuda)
print("gpu count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("cap:", torch.cuda.get_device_capability(0))


torch: 2.5.1
cuda available: True
torch cuda version: 12.1
gpu count: 1
gpu: NVIDIA GeForce RTX 3060 Ti
cap: (8, 6)


In [11]:
# Cell 8 — Train
# ---------------------------------------------------------------
train_result = trainer.train()
train_result


  2%|▏         | 50/2080 [00:21<17:15,  1.96it/s]

{'loss': 0.8215, 'learning_rate': 7.04e-06, 'epoch': 0.48}


  5%|▍         | 100/2080 [00:39<13:28,  2.45it/s]

{'loss': 0.7419, 'learning_rate': 1.4720000000000001e-05, 'epoch': 0.96}


  5%|▌         | 104/2080 [00:40<09:04,  3.63it/s]c:\Users\USER\miniconda3\envs\torch-gpu\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
                                                  
  5%|▌         | 104/2080 [00:42<09:04,  3.63it/s]

...

                                                   
100%|██████████| 2080/2080 [22:54<00:00,  2.88it/s]

{'eval_loss': 0.693351149559021, 'eval_accuracy': 0.5, 'eval_precision_pos': 0.5, 'eval_recall_pos': 0.8876404494382022, 'eval_f1_pos': 0.6396761133603239, 'eval_tn': 10, 'eval_fp': 79, 'eval_fn': 10, 'eval_tp': 79, 'eval_runtime': 1.1066, 'eval_samples_per_second': 160.86, 'eval_steps_per_second': 20.785, 'epoch': 20.0}


100%|██████████| 2080/2080 [23:49<00:00,  1.46it/s]

{'train_runtime': 1429.0865, 'train_samples_per_second': 11.56, 'train_steps_per_second': 1.455, 'train_loss': 0.713957405090332, 'epoch': 20.0}


TrainOutput(global_step=2080, training_loss=0.713957405090332, metrics={'train_runtime': 1429.0865, 'train_samples_per_second': 11.56, 'train_steps_per_second': 1.455, 'train_loss': 0.713957405090332, 'epoch': 20.0})

In [12]:
# Cell 9 — Evaluate on validation + test
# ---------------------------------------------------------------
print("Best checkpoint:", trainer.state.best_model_checkpoint)

val_metrics = trainer.evaluate(val_tok)
print("\nVAL metrics:")
for k, v in val_metrics.items():
    if k.startswith("eval_"):
        print(f"{k}: {v}")

test_metrics = trainer.evaluate(test_tok)
print("\nTEST metrics:")
for k, v in test_metrics.items():
    if k.startswith("eval_"):
        print(f"{k}: {v}")


Best checkpoint: runs_tashbeeh/aubmindlab_bert-large-arabertv2\checkpoint-728


100%|██████████| 23/23 [00:00<00:00, 23.61it/s]



VAL metrics:
eval_loss: 0.6970489025115967
eval_accuracy: 0.5
eval_precision_pos: 0.5
eval_recall_pos: 1.0
eval_f1_pos: 0.6666666666666666
eval_tn: 0
eval_fp: 89
eval_fn: 0
eval_tp: 89
eval_runtime: 1.1471
eval_samples_per_second: 155.18
eval_steps_per_second: 20.051


100%|██████████| 23/23 [00:00<00:00, 23.38it/s]


TEST metrics:
eval_loss: 0.6954743266105652
eval_accuracy: 0.5
eval_precision_pos: 0.5
eval_recall_pos: 1.0
eval_f1_pos: 0.6666666666666666
eval_tn: 0
eval_fp: 89
eval_fn: 0
eval_tp: 89
eval_runtime: 1.0295
eval_samples_per_second: 172.898
eval_steps_per_second: 22.341


In [13]:
# Cell 10 — Predict on test set + show some errors (false positives/negatives)
# ---------------------------------------------------------------
pred_out = trainer.predict(test_tok)
test_logits = pred_out.predictions
test_preds = np.argmax(test_logits, axis=-1)
test_labels = pred_out.label_ids

# Rebuild a viewable df aligned with test_df order
view_df = pd.DataFrame({
    TEXT_COL: test_ds[TEXT_COL],
    "y_true": test_labels,
    "y_pred": test_preds,
})

# Show false negatives (missed tashbeeh)
fn_df = view_df[(view_df["y_true"] == 1) & (view_df["y_pred"] == 0)].head(15)
fp_df = view_df[(view_df["y_true"] == 0) & (view_df["y_pred"] == 1)].head(15)

print("False Negatives (missed tashbeeh):", fn_df.shape[0])
display(fn_df)

print("False Positives (predicted tashbeeh but actually non):", fp_df.shape[0])
display(fp_df)


100%|██████████| 23/23 [00:00<00:00, 23.62it/s]


False Negatives (missed tashbeeh): 0


,البيت_الكامل,y_true,y_pred


False Positives (predicted tashbeeh but actually non): 15


,البيت_الكامل,y_true,y_pred
1,ومهمهٍ نازحٍ ، تعوي الذئابُ به، نائي المياهِ ع...,0,1
2,بَلْ عَجِبُوا أَنْ جَاءَهُمْ مُنْذِرٌ مِنْهُمْ...,0,1
4,فإن أكُ مظلوماً؛ فعبدٌ ظلمتَهُ؛ وإن تكُ ذا عُت...,0,1
6,لأْيًا يُجَاهِدُهَا لا يَأْتَلِي طَلَبًا لأْيً...,0,1
7,كَرِيمًا حِينَ يُنْسَبُ والِدَاهُ إلَى كَرَمِ ...,0,1
12,يا طَلْعَةً طَلَعَ الحِمامُ عليها وجنى لها ثَم...,0,1
13,لم يَبْقَ لِلصَّيْفِ لا رَسْمٌ ولا طَلَلُ ولا ...,0,1
15,حَكَمَتْ لأَنْفُسِها اللَّيالي أَنَّها أَبداً ...,0,1
17,إِنَّا نَحْنُ نُحْيِي وَنُمِيتُ وَإِلَيْنَا ال...,0,1
19,وَكُنَّا إِذَا ما خَانَنَا الدَّهْرُ أوْ سَرَى...,0,1


In [14]:
# Cell 11 — Save the best model + tokenizer (for later inference)
# ---------------------------------------------------------------
BEST_DIR = os.path.join(OUTPUT_DIR, "best_model")
trainer.save_model(BEST_DIR)
tokenizer.save_pretrained(BEST_DIR)
print("Saved to:", BEST_DIR)


Saved to: runs_tashbeeh/aubmindlab_bert-large-arabertv2\best_model


In [15]:
# Cell 12 — Quick inference function (single verse)
# ---------------------------------------------------------------
import torch

def predict_tashbeeh(verse: str) -> int:
    model.eval()
    enc = tokenizer(verse, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(device)
    with torch.no_grad():
        logits = model(**enc).logits
    pred = int(torch.argmax(logits, dim=-1).item())
    return pred  # 1=tashbeeh, 0=non

# Example:
# print(predict_tashbeeh("أهُمُّ بِشَيْءٍ واللَّيالي كأنَّهَا تُطارِدُنِي"))
